In [1]:
import sys
sys.path.append('../')

import numpy as np
from openfermion import (
    QubitOperator, 
    get_sparse_operator
)
from utils_m2_factorize import (
    expand_tensor_product, 
    expand_tensor_product_for_incomplete_qubit_set,
    partial_trace_einsum
)
from numpy.random import uniform

Test 1: prepare a state $|\psi\rangle$ which is a tensor product state

$$
|\psi\rangle = |\psi_{012}\rangle \otimes |\psi_{345}\rangle.
$$

I should be able to recover $|\psi\rangle$ from $|\psi_{012}\rangle$ and $|\psi_{345}\rangle$ using the `expand_tensor_product` functions. I should be able to recover $|\psi_{012}\rangle$ and $|\psi_{345}\rangle$ from $|\psi\rangle$ using the `partial_trace_einsum` function.

In [ ]:
for _ in range(100):
    H012 = QubitOperator()

    val   = uniform(-1, 1)
    H012 += val * QubitOperator('X0 X2')

    val   = uniform(-1, 1)
    H012 += val * QubitOperator('X0 Z1 Z2')

    val   = uniform(-1, 1)
    H012 += val * QubitOperator('Y0 Y1 X2')

    val   = uniform(-1, 1)
    H012 += val * QubitOperator('Z2')

    val   = uniform(-1, 1)
    H012 += val * QubitOperator('Z0 Z1 Z2')


    H345  = QubitOperator()
    H345b = QubitOperator()

    val    = uniform(-1, 1)
    H345  += val * QubitOperator('X3 X4')
    H345b += val * QubitOperator('X0 X1')

    val    = uniform(-1, 1)
    H345  += val * QubitOperator('X3 X4 Z5')
    H345b += val * QubitOperator('X0 X1 Z2')

    val    = uniform(-1, 1)
    H345  += val * QubitOperator('Y3 Y5')
    H345b += val * QubitOperator('Y0 Y2')

    val    = uniform(-1, 1)
    H345  += val * QubitOperator('Z4')
    H345b += val * QubitOperator('Z1')

    val    = uniform(-1, 1)
    H345  += val * QubitOperator('X3 Z4 X5')
    H345b += val * QubitOperator('X0 Z1 X2')


    H012345 = H012 * H345

    H012345sparse = get_sparse_operator(H012345, 6)
    H012sparse    = get_sparse_operator(H012, 3)
    H345bsparse   = get_sparse_operator(H345b, 3)

    # create full state and factored states

    psi_init                   = np.zeros(2 ** 6)
    psi_init[int('110001', 2)] = 1
    psi                        = (H012345sparse @ psi_init) / np.linalg.norm(H012345sparse @ psi_init)


    psi012_init                = np.zeros(2 ** 3)
    psi012_init[int('110', 2)] = 1
    psi012                     = (H012sparse @ psi012_init) / np.linalg.norm(H012sparse @ psi012_init)

    psi345_init                = np.zeros(2 ** 3)
    psi345_init[int('001', 2)] = 1
    psi345                     = (H345bsparse @ psi345_init) / np.linalg.norm(H345bsparse @ psi345_init)

    factorization_dict = {
        (0,1,2) : psi012,
        (3,4,5) : psi345
    }

    psi_recov1 = np.kron(psi012, psi345)
    psi_recov2 = expand_tensor_product(factorization_dict, 6)
    psi_recov3 = expand_tensor_product_for_incomplete_qubit_set(factorization_dict)

    assert np.allclose(psi, psi_recov1)
    assert np.allclose(psi, psi_recov2)
    assert np.allclose(psi, psi_recov3)

    psi012_recov, check012     = partial_trace_einsum(psi, (0,1,2), 6)
    psi345_recov, check345     = partial_trace_einsum(psi, (3,4,5), 6)
    psi0145_attempt, check0145 = partial_trace_einsum(psi, (0,1,4,5), 6) # state doesn't factorize over 0145 so check0145 should be false

    assert check012
    assert check345 
    assert not check0145

    assert np.allclose(psi012, psi012_recov) or np.allclose(psi012, -psi012_recov)
    assert np.allclose(psi345, psi345_recov) or np.allclose(psi345, -psi345_recov)



Test 2: prepare a state $|\psi\rangle$ which is a tensor product state on non-adjacent qubits

$$
|\psi\rangle = |\psi_{034}\rangle \otimes |\psi_{126}\rangle \otimes |\psi_5\rangle.
$$

I should be able to recover $|\psi\rangle$ from $|\psi_{034}\rangle, |\psi_{126}\rangle, |\psi_5\rangle$ using the `expand_tensor_product` functions. Here, `np.kron` cannot be used due to non-adjacency of the qubit sets. I should also be able to recover the factors from $|\psi\rangle$ using `partial_trace_einsum`.

In [37]:
for _ in range(100):

    H034  = QubitOperator()
    H034b = QubitOperator()

    val    = uniform(-1, 1)
    H034  += val * QubitOperator('X0 X3')
    H034b += val * QubitOperator('X0 X1')

    val    = uniform(-1, 1)
    H034  += val * QubitOperator('X0 X3 Z4')
    H034b += val * QubitOperator('X0 X1 Z2')

    val    = uniform(-1, 1)
    H034  += val * QubitOperator('Y0 Y4')
    H034b += val * QubitOperator('Y0 Y2')

    val    = uniform(-1, 1)
    H034  += val * QubitOperator('Z3')
    H034b += val * QubitOperator('Z1')

    val    = uniform(-1, 1)
    H034  += val * QubitOperator('X0 Z3 X4')
    H034b += val * QubitOperator('X0 Z1 X2')

    H126  = QubitOperator()
    H126b = QubitOperator()

    val    = uniform(-1, 1)
    H126  += val * QubitOperator('X1 X6')
    H126b += val * QubitOperator('X0 X2')

    val    = uniform(-1, 1)
    H126  += val * QubitOperator('Z1 X2 Z6')
    H126b += val * QubitOperator('Z0 X1 Z2')

    val    = uniform(-1, 1)
    H126  += val * QubitOperator('Y1 X2 Y6')
    H126b += val * QubitOperator('Y0 X1 Y2')

    val    = uniform(-1, 1)
    H126  += val * QubitOperator('Z2')
    H126b += val * QubitOperator('Z1')

    val    = uniform(-1, 1)
    H126  += val * QubitOperator('X1 Z2 X6')
    H126b += val * QubitOperator('X0 Z1 X2')

    val    = uniform(-1, 1)
    H126  += val * QubitOperator('Z1 Z2')
    H126b += val * QubitOperator('Z0 Z1')

    H5  = QubitOperator()
    H5b = QubitOperator()

    val = uniform(-1, 1)
    H5  += val * QubitOperator('X5')
    H5b += val * QubitOperator('X0')

    val = uniform(-1, 1)
    H5  += val * QubitOperator('Z5')
    H5b += val * QubitOperator('Z0')

    val = uniform(-1, 1)
    H5  += val * QubitOperator('')
    H5b += val * QubitOperator('') 

    H0123456 = H034 * H126 * H5

    H0123456sparse = get_sparse_operator(H0123456, 7)
    H034bsparse    = get_sparse_operator(H034b, 3)
    H126bsparse    = get_sparse_operator(H126b, 3)
    H5bsparse      = get_sparse_operator(H5b, 1)

    psi_init                    = np.zeros(2 ** 7)
    psi_init[int('1100101', 2)] = 1
    psi                         = (H0123456sparse @ psi_init) / np.linalg.norm(H0123456sparse @ psi_init)

    psi034init                  = np.zeros(2 ** 3)
    psi034init[int('101', 2)]   = 1
    psi034                      = (H034bsparse @ psi034init) / np.linalg.norm(H034bsparse @ psi034init)

    psi126init                  = np.zeros(2 ** 3)
    psi126init[int('101', 2)]   = 1
    psi126                      = (H126bsparse @ psi126init) / np.linalg.norm(H126bsparse @ psi126init)

    psi5init                    = np.zeros(2 ** 1)
    psi5init[int('0', 2)]       = 1
    psi5                        = (H5bsparse @ psi5init) / np.linalg.norm(H5bsparse @ psi5init)

    factorization_dict = {
        (0,3,4) : psi034,
        (1,2,6) : psi126,
        (5,)    : psi5
    }

    psi_recov1 = expand_tensor_product(factorization_dict, 7)
    psi_recov2 = expand_tensor_product_for_incomplete_qubit_set(factorization_dict)

    assert np.allclose(psi, psi_recov1)
    assert np.allclose(psi, psi_recov2)

    psi034_recov, check034 = partial_trace_einsum(psi, (0,3,4), 7)
    psi126_recov, check126 = partial_trace_einsum(psi, (1,2,6), 7)
    psi5_recov, check5     = partial_trace_einsum(psi, (5,), 7)
    psi01_attempt, check01 = partial_trace_einsum(psi, (0,1), 7)

    assert check034
    assert check126
    assert check5
    assert not check01

    assert np.allclose(psi034, psi034_recov) or np.allclose(psi034, -psi034_recov)
    assert np.allclose(psi126, psi126_recov) or np.allclose(psi126, -psi126_recov)
    assert np.allclose(psi5, psi5_recov) or np.allclose(psi5, -psi5_recov)

Test 3: Here, we test the `expand_tensor_product_for_incomplete_qubit_set` function. We use the same code as last time. We should get the correct state no matter what the keys of `factorization_dict` are, as long as the changes I make preserve order relations.

In [5]:
H034  = QubitOperator()
H034b = QubitOperator()

val    = uniform(-1, 1)
H034  += val * QubitOperator('X0 X3')
H034b += val * QubitOperator('X0 X1')

val    = uniform(-1, 1)
H034  += val * QubitOperator('X0 X3 Z4')
H034b += val * QubitOperator('X0 X1 Z2')

val    = uniform(-1, 1)
H034  += val * QubitOperator('Y0 Y4')
H034b += val * QubitOperator('Y0 Y2')

val    = uniform(-1, 1)
H034  += val * QubitOperator('Z3')
H034b += val * QubitOperator('Z1')

val    = uniform(-1, 1)
H034  += val * QubitOperator('X0 Z3 X4')
H034b += val * QubitOperator('X0 Z1 X2')

H126  = QubitOperator()
H126b = QubitOperator()

val    = uniform(-1, 1)
H126  += val * QubitOperator('X1 X6')
H126b += val * QubitOperator('X0 X2')

val    = uniform(-1, 1)
H126  += val * QubitOperator('Z1 X2 Z6')
H126b += val * QubitOperator('Z0 X1 Z2')

val    = uniform(-1, 1)
H126  += val * QubitOperator('Y1 X2 Y6')
H126b += val * QubitOperator('Y0 X1 Y2')

val    = uniform(-1, 1)
H126  += val * QubitOperator('Z2')
H126b += val * QubitOperator('Z1')

val    = uniform(-1, 1)
H126  += val * QubitOperator('X1 Z2 X6')
H126b += val * QubitOperator('X0 Z1 X2')

val    = uniform(-1, 1)
H126  += val * QubitOperator('Z1 Z2')
H126b += val * QubitOperator('Z0 Z1')

H5  = QubitOperator()
H5b = QubitOperator()

val = uniform(-1, 1)
H5  += val * QubitOperator('X5')
H5b += val * QubitOperator('X0')

val = uniform(-1, 1)
H5  += val * QubitOperator('Z5')
H5b += val * QubitOperator('Z0')

val = uniform(-1, 1)
H5  += val * QubitOperator('')
H5b += val * QubitOperator('') 

H0123456 = H034 * H126 * H5

H0123456sparse = get_sparse_operator(H0123456, 7)
H034bsparse    = get_sparse_operator(H034b, 3)
H126bsparse    = get_sparse_operator(H126b, 3)
H5bsparse      = get_sparse_operator(H5b, 1)

psi_init                    = np.zeros(2 ** 7)
psi_init[int('1100101', 2)] = 1
psi                         = H0123456sparse @ psi_init

psi034init                = np.zeros(2 ** 3)
psi034init[int('101', 2)] = 1
psi034                    = H034bsparse @ psi034init

psi126init                = np.zeros(2 ** 3)
psi126init[int('101', 2)] = 1
psi126                    = H126bsparse @ psi126init

psi5init              = np.zeros(2 ** 1)
psi5init[int('0', 2)] = 1
psi5                  = H5bsparse @ psi5init


# unchanged from previous 

factorization_dict = {
    (0,3,4) : psi034,
    (1,2,6) : psi126,
    (5,)    : psi5
}

psi_recov2 = expand_tensor_product_for_incomplete_qubit_set(factorization_dict)
assert np.allclose(psi, psi_recov2)

# changed values which preserve order

factorization_dict = {
    (0,10,17) : psi034,
    (3,7,23)  : psi126,
    (19,)     : psi5
}

psi_recov3 = expand_tensor_product_for_incomplete_qubit_set(factorization_dict)
assert np.allclose(psi, psi_recov3)

# changed values which do not preserve order

factorization_dict = {
    (0,7,17)  : psi034,
    (3,10,23) : psi126,
    (19,)     : psi5
}

psi_recov4 = expand_tensor_product_for_incomplete_qubit_set(factorization_dict)
assert not np.allclose(psi, psi_recov4)

